[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/10_gqa_solution.ipynb)

# ✅ Solution: gqa

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> jax.Array:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp, math
from flax import nnx
class GroupQueryAttention(nnx.Module):
    """B=batch, L=seq, D=d_model, H=query heads, G=KV heads, K=d_k."""
    def __init__(self, d_model, num_heads, num_kv_heads, *, rngs):
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.W_q = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_k = nnx.Linear(d_model, num_kv_heads * self.d_k, rngs=rngs)
        self.W_v = nnx.Linear(d_model, num_kv_heads * self.d_k, rngs=rngs)
        self.W_o = nnx.Linear(d_model, d_model, rngs=rngs)
    def __call__(self, x_BLD):
        B, L, _ = x_BLD.shape
        H, G, K = self.num_heads, self.num_kv_heads, self.d_k
        q_BHLK = self.W_q(x_BLD).reshape(B, L, H, K).transpose(0, 2, 1, 3)
        k_BGLK = self.W_k(x_BLD).reshape(B, L, G, K).transpose(0, 2, 1, 3)
        v_BGLK = self.W_v(x_BLD).reshape(B, L, G, K).transpose(0, 2, 1, 3)
        k_BHLK = jnp.repeat(k_BGLK, H // G, axis=1)
        v_BHLK = jnp.repeat(v_BGLK, H // G, axis=1)
        scores_BHLL = q_BHLK @ jnp.swapaxes(k_BHLK, -2, -1) / math.sqrt(K)
        attn_BHLK = jax.nn.softmax(scores_BHLL, axis=-1) @ v_BHLK
        concat_BLD = attn_BHLK.transpose(0, 2, 1, 3).reshape(B, L, H * K)
        return self.W_o(concat_BLD)


In [ ]:
# Verify
print(GroupQueryAttention)


In [ ]:
from jax_judge import check
check("gqa")
